In [1]:
import os
import csv
import netCDF4 as netcdf
import xarray as xr
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shutil
import datetime
import re

In [3]:
#Timestamp filtering

indirectory = '/Users/jennybrun/Documents/Skole/DTU/Skole/Fall25/Remote Sensing/RemoteSensingProject/seaicemapping/inputs/SCAMS_colocated/'
outdirectory = '/Users/jennybrun/Documents/Skole/DTU/Skole/Fall25/Remote Sensing/RemoteSensingProject/seaicemapping/SCAMS_fixed/'

if os.path.exists(outdirectory):
    shutil.rmtree(outdirectory)

os.makedirs(outdirectory, exist_ok=True)

good_orbit_times =  []
bad_orbit_times = []
files = [f for f in os.listdir(indirectory) if f.endswith('.nc')]


good_files = []
bad_files = []
unknown_errors = []

for fname in files:
    
    ds = xr.open_dataset("inputs/SCAMS_colocated/" + fname)
    hours = pd.to_datetime(ds.Time.values).hour
    nunique_hours = hours.nunique()
    unique_hours = hours.unique()
    #print(unique_hours)
    #print(fname)

    if hours.nunique() > 3 or (unique_hours[0] == 0 and unique_hours[1] != 1): 
        if (unique_hours[0] == 0 and unique_hours[1] != 1):
            ds = ds.isel(Time=slice(1, None))
            
        else:
            bad_files.append(fname)
            ds.close()
            tbch1 = ds.TBCH1
            timestamps = tbch1.Time
    #new_hours = pd.to_datetime(ds.Time.values).hour
            orbit_time = int((timestamps[-1] - timestamps[0])/1e9/60) 
            bad_orbit_times.append(orbit_time)
            continue
    
    tbch1 = ds.TBCH1
    timestamps = tbch1.Time
    #new_hours = pd.to_datetime(ds.Time.values).hour
    orbit_time = int((timestamps[-1] - timestamps[0])/1e9/60) 

    if orbit_time > 107:
        unknown_errors.append(orbit_time)
        #print("unknown error orbit time too long")
        bad_files.append(fname)
        ds.close()
        bad_orbit_times.append(orbit_time)
        continue

    else:
        good_orbit_times.append(orbit_time)

        
    outpath = os.path.join(outdirectory, os.path.basename(fname))
    ds.to_netcdf(outpath)
    good_files.append(fname)






In [5]:
#Brightness temperature filtering

indir = '/Users/jennybrun/Documents/Skole/DTU/Skole/Fall25/Remote Sensing/RemoteSensingProject/seaicemapping/SCAMS_fixed/'
outdir = '/Users/jennybrun/Documents/Skole/DTU/Skole/Fall25/Remote Sensing/RemoteSensingProject/seaicemapping/SCAMS_tbfixed/'

if os.path.exists(outdir):
    shutil.rmtree(outdir)

os.makedirs(outdir, exist_ok=True)

fixed_files = [f for f in os.listdir(indir) if f.endswith('.nc')]
good_f = []
bad_f = []

for fname in fixed_files:
    ds = xr.open_dataset("SCAMS_fixed/" + fname)
    tbch1 = np.asarray(ds.TBCH1)
    tbch2 = np.asarray(ds.TBCH2)
    error_in_file = False

    for i in range(13):
        tbch1_angle = tbch1[:, i]
        tbch2_angle = tbch2[:, i]

        tbch1_vals, tbch1_counts = np.unique(tbch1_angle, return_counts=True)
        tbch1_max_count = tbch1_counts.max()

        if tbch1_max_count > tbch1_angle.size*0.25:
            #print("Error in the brightness temp in Tb channel 1")
            error_in_file = True
            continue
        tbch2_vals, tbch2_counts = np.unique(tbch2_angle, return_counts=True)
        tbch2_max_count = tbch2_counts.max()

        if tbch2_max_count > tbch2_angle.size*0.25:
            #print("Error in the brightness temp in Tb channel 2")
            error_in_file = True
            continue

    if error_in_file == False:
        outpath = os.path.join(outdir, os.path.basename(fname))
        ds.to_netcdf(outpath)
        good_f.append(fname)

    else:
        bad_f.append(fname)
        continue




In [4]:
last_longitude = np.asarray(first_ds.LON)
last_latitude = np.asarray(first_ds.LAT)


for f in files_sorted[:3]:
    ds = xr.open_dataset("SCAMS_fixed/" + f)
    this_longitude = ds.LON
    this_latitude = ds.LAT

    plt.scatter(ds.LON.values.flatten(), ds.LAT.values.flatten(), s=1)
    plt.title("SCAMS swath geometry")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")

NameError: name 'first_ds' is not defined

In [ ]:
print(last_latitude[0,0])
print(last_latitude[0,1])
print(last_latitude[0,2])
print(last_latitude[0,3])
print(last_latitude[0,12])



-46.7812
-46.4688
-46.125
-45.8125
-41.25


**Filtering:**
I am first filtering out error in timestamp, then i go through the new directory to filter out error in brightness temperature. Should maybe combine these two later...

1. Timestamps

    Assess the time stamps of the data. Especially the first time stamp seems to be prone to errors. A single file should only cover 107 minutes (one orbit). A regular file should therefore contain a maximum of 3 different hour-stamps. 
    First i check if the file has more than 3 different hour-stamps or if the first unique hour-stamp is 0 and the second unique hour stamp is not one. If one of these is true, something is wrong with the data. If the first unique hour stamp is 0 and the second is not 1, i slice away the data corresponding to the first time stamp. If that is not the case we know that the data has more than 3 unique hour stamps and we want to discard the file. I add the bad file to bad_files to keep track of the discarded files. Next i calculate the orbit-time. I found that some files (30 files) had less than three hour stamps and did not have a 0 for the first time-stamp, but the orbit time was still over 107 minutes (some were 4000+ min), so i discard these as well and add the file to bad_files. If none of the above is the case the orbit time is added to orbit_times and the file is added to the new directory SCAMS_fixed which contains only good_files. This resulted in 317 good files and 54 bad files. 

2. Brightness Temperature

    Check if one of the two lowest brighness temperature channels has stopped at a constant value and does not represent physical surface values. Need to check each beam position and see if more than 25% of the datapoints have the exact same Tb value. I only found errors on channel 2. 15 files had bad Tb data, i have for now discarded these 15 files, not sure if that is correct. 

3. Overlapping records

    I start with making sure the files are in the right orber by datetime since we are looking for overlap between two subsequent files. We have 13 beams but I wonder if it is safe to check for overlap at one beam and assume that if there is an overlap at that beam there is an overlap at the other beams? (We dont need to do this:)




In [ ]:
orbit_time = int((t[-1] - t[1])/1000000000/60) 
print(f'orbit time: {orbit_time} minutes')
angles = [-53.3, -43.5, -34.4, -25.6, -16.9, -8.4, 0.0, 8.4, 16.9, 25.6, 34.4, 53.3]
fig = plt.figure(figsize= (18, 8))
for i in range(len(angles)):
    tbch_1 = tbch1[:, i]
    plt.plot(t[1:], tbch_1[1:], "o", label="Angle: " + str(angles[i]) + " deg")
    plt.legend()

plt.title("TB from Channel 1")


NameError: name 't' is not defined